<a href="https://colab.research.google.com/github/Aswin-k61/NLP_Repo/blob/main/Neural_network_tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import nltk
import re
import string

from google.colab import files
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

nltk.download('stopwords')



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
df=pd.read_csv('/content/IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [4]:
df.shape

(50000, 2)

In [5]:
df.isnull().sum()

,0
review,0
sentiment,0


In [6]:
df=df.sample(
    5000,
    random_state=42
)

In [7]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
}
)

In [8]:
from nltk.corpus.reader import WordNetCorpusReader
stop_words=set(stopwords.words('english'))
stemmer=PorterStemmer()

def preprocess(text):
  text=text.lower()
  text=re.sub('<.*?>','',text)

 # remove punctuation
  text=text.translate(
  str.maketrans('','',string.punctuation))

 # tokenization
  words=text.split()

 # stopword removal
  words=[
    word for word in words
    if word not in stop_words
  ]

  #stemming
  words=[
      stemmer.stem(word)
      for word in words
   ]


  return " ".join(words)


In [9]:
df['clean_review']=df['review'].apply(preprocess)

In [10]:
vectorizer=TfidfVectorizer(
    max_features=5000,
)
X=vectorizer.fit_transform(df['clean_review']).toarray()
y=df['sentiment']

In [11]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [12]:
model=Sequential()
model.add(Dense(128,activation='relu',input_shape=(X_train.shape[1],)))
model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [14]:
model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
)

Epoch 1/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7598 - loss: 0.5002
Epoch 2/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9377 - loss: 0.1654
Epoch 3/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9898 - loss: 0.0475
Epoch 4/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9992 - loss: 0.0108
Epoch 5/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 1.0000 - loss: 0.0031
Epoch 6/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 1.0000 - loss: 0.0016
Epoch 7/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 1.0000 - loss: 0.0010
Epoch 8/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 6.8676e-04
Epoch 9/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 1.0000 - loss: 4.9677e-04
Epoch 10/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 1.0000 - loss: 3.7153e-04
Epoch 11/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 1.0000 - loss: 2.8709e-04
Epoch 12/15
125/125 ━━━━━━━━━━━━

In [15]:
loss,accuracy=model.evaluate(X_test,y_test)
print("Accuracy:",accuracy)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8220 - loss: 0.9034
Accuracy: 0.8220000267028809


In [16]:
review=["This movie was fantastic"]
clean=preprocess(review[0])
vector=vectorizer.transform(
    [clean]
).toarray()
pred=model.predict(vector)
if pred>0.5:
  print("Positive")
else:
  print("Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
Positive


In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X1)

sequences = tokenizer.texts_to_sequences(X1)

In [18]:
X1 = df['clean_review']      # your text
y1 = df['sentiment']        # 0 or 1

In [20]:
max_len = 100

X_pad = pad_sequences(sequences, maxlen=max_len, padding='post')

In [21]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X_pad, y, test_size=0.2, random_state=42
)

In [22]:
model1 = Sequential()

model1.add(Embedding(
    input_dim=5000,
    output_dim=32,
    input_length=max_len
))

model1.add(SimpleRNN(32))   # ✅ Simple RNN only

model1.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [23]:
model1.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [39]:
model1.fit(
    X_train1,
    y_train1,
    epochs=15,
    batch_size=32,
    validation_data=(X_test1, y_test1)
)

Epoch 1/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 1.0000 - loss: 7.5253e-04 - val_accuracy: 0.4930 - val_loss: 1.5504
Epoch 2/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 1.0000 - loss: 6.4759e-04 - val_accuracy: 0.4960 - val_loss: 1.5746
Epoch 3/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 1.0000 - loss: 5.6163e-04 - val_accuracy: 0.4980 - val_loss: 1.5958
Epoch 4/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 1.0000 - loss: 4.9106e-04 - val_accuracy: 0.4940 - val_loss: 1.6140
Epoch 5/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 1.0000 - loss: 4.3184e-04 - val_accuracy: 0.4970 - val_loss: 1.6392
Epoch 6/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 1.0000 - loss: 3.8103e-04 - val_accuracy: 0.4950 - val_loss: 1.6578
Epoch 7/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 1.0000 - loss: 3.3867e-04 - val_accuracy: 0.4960 - val_loss: 1.6724
Epoch 8/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 1.00

In [40]:
loss, acc = model1.evaluate(X_test1, y_test1)
print("Test Accuracy:", acc)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4960 - loss: 1.8029
Test Accuracy: 0.4959999918937683


In [27]:
X2 = df['clean_review']      # your text
y2 = df['sentiment']

In [28]:
from tensorflow.keras.layers import LSTM

In [29]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X2)

sequences = tokenizer.texts_to_sequences(X2)

In [30]:
max_len = 100

X_pad = pad_sequences(sequences, maxlen=max_len, padding='post')

In [31]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_pad, y, test_size=0.2, random_state=42
)

In [36]:
model2 = Sequential()

model2.add(Embedding(
    input_dim=5000,     # vocab size
    output_dim=64,      # embedding size
    input_length=max_len
))

model2.add(LSTM(64))     # 🔥 better than SimpleRNN

model2.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [37]:
model2.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [41]:
model2.fit(
    X_train2,
    y_train2,
    epochs=15,
    batch_size=32,
    validation_data=(X_test2, y_test2)
)

Epoch 1/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 49ms/step - accuracy: 0.8735 - loss: 0.3528 - val_accuracy: 0.7430 - val_loss: 0.6469
Epoch 2/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 56ms/step - accuracy: 0.8648 - loss: 0.3585 - val_accuracy: 0.7810 - val_loss: 0.5349
Epoch 3/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 47ms/step - accuracy: 0.8965 - loss: 0.2814 - val_accuracy: 0.8170 - val_loss: 0.5486
Epoch 4/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 10s 47ms/step - accuracy: 0.8810 - loss: 0.3705 - val_accuracy: 0.8000 - val_loss: 0.4980
Epoch 5/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 58ms/step - accuracy: 0.9153 - loss: 0.2475 - val_accuracy: 0.8120 - val_loss: 0.5108
Epoch 6/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 46ms/step - accuracy: 0.9160 - loss: 0.2149 - val_accuracy: 0.8100 - val_loss: 0.5116
Epoch 7/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 58ms/step - accuracy: 0.9367 - loss: 0.1824 - val_accuracy: 0.7960 - val_loss: 0.5323
Epoch 8/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 46ms/step - accuracy: 0.9405 - loss: 0.1766 - val_acc

In [42]:
loss, acc = model.evaluate(X_test2, y_test2)
print("Test Accuracy:", acc)

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5630 - loss: 0.6722
Test Accuracy: 0.5630000233650208
